# KonkaniVani ASR - Complete Retraining (FIXED)

## What's Fixed:
- ✅ CTC weight increased to 0.8 (was 0.3)
- ✅ Using full 88-hour dataset (was 21h)
- ✅ Periodic testing every 5 epochs
- ✅ Better learning rate and gradient clipping

## Expected Results:
- Epoch 20: Blank prob < 80% (model starts working)
- Epoch 40: Blank prob < 60% (good transcriptions)
- Epoch 100: Blank prob < 40% (production ready)

## Step 1: Setup Environment

In [ ]:
# Install dependencies
!pip install -q torch torchaudio librosa soundfile jiwer pyyaml tensorboard matplotlib

# Suppress dependency warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import sys
import json
import torch
import torchaudio
from pathlib import Path
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Check Dataset

In [ ]:
# List available datasets
!ls -lh /kaggle/input/

In [ ]:
# Set paths - UPDATE THESE to match your dataset names
from pathlib import Path

DATA_ROOT = Path('/kaggle/input/konkani-asr-complete-data')  # Your main data
SCRIPTS_ROOT = Path('/kaggle/input/kaggle-training-scripts')  # Your scripts dataset

print(f"Data dataset: {DATA_ROOT}")
print(f"Scripts dataset: {SCRIPTS_ROOT}")
print("\nDataset structure:")
!ls -lh {DATA_ROOT}
print("\nScripts structure:")
!ls -lh {SCRIPTS_ROOT}

## Step 3: Extract and Prepare Data

In [ ]:
# Copy training scripts from scripts dataset
import shutil
import zipfile

# Check if datasets exist
if not SCRIPTS_ROOT.exists():
    print(f"✗ ERROR: Scripts dataset not found at {SCRIPTS_ROOT}")
    print("\nPlease add the 'kaggle-training-scripts' dataset as input to this notebook.")
    print("Click 'Add Input' → Search for your scripts dataset → Add it")
    raise FileNotFoundError(f"Scripts dataset not found: {SCRIPTS_ROOT}")

if not DATA_ROOT.exists():
    print(f"✗ ERROR: Data dataset not found at {DATA_ROOT}")
    print("\nPlease add your data dataset as input to this notebook.")
    print("Click 'Add Input' → Search for your data dataset → Add it")
    print("\nAvailable datasets:")
    !ls -la /kaggle/input/
    raise FileNotFoundError(f"Data dataset not found: {DATA_ROOT}")

print("✓ Both datasets found!")
print("\nCopying training scripts...")
script_dirs = ['training_scripts', 'models', 'scripts', 'data']

for dir_name in script_dirs:
    src_dir = SCRIPTS_ROOT / 'tmp' / 'kaggle_scripts_package' / dir_name
    if src_dir.exists():
        dst_dir = Path('/kaggle/working') / dir_name
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        print(f"  ✓ Copied {dir_name}/")
    else:
        print(f"  ⚠️  {dir_name}/ not found at {src_dir}")

print("\n✓ Training scripts ready!")

# Extract data files if zipped
zip_files = list(DATA_ROOT.glob('*.zip'))
if zip_files:
    print(f"\nFound {len(zip_files)} data zip files. Extracting...")
    for zip_file in zip_files:
        print(f"Extracting {zip_file.name}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall('/kaggle/working/')
    print("✓ Data extraction complete!")
else:
    print("\nNo data zip files found, copying data directly...")
    # Copy data files directly if not zipped
    for item in DATA_ROOT.iterdir():
        if item.is_dir():
            dst = Path('/kaggle/working') / item.name
            if not dst.exists():
                shutil.copytree(item, dst)
                print(f"  ✓ Copied {item.name}/")

In [ ]:
# Verify extracted files
print("Checking extracted structure...")
print("\nPython scripts:")
!ls -lh /kaggle/working/training_scripts/*.py 2>/dev/null || echo "  ✗ training_scripts not found"
!ls -lh /kaggle/working/models/*.py 2>/dev/null || echo "  ✗ models not found"
!ls -lh /kaggle/working/scripts/*.py 2>/dev/null || echo "  ✗ scripts not found"

print("\nData files:")
!ls -lh /kaggle/working/ | head -15

## Step 4: Locate Dataset Manifests

In [ ]:
# Find manifest files in the extracted data
import os
import json

# Search for manifest directories
possible_manifest_dirs = [
    '/kaggle/working/konkani-10k',  # Your dataset location
    '/kaggle/working/data/konkani-combined/manifests',
    '/kaggle/working/data/konkani-asr-v0/splits/manifests',
    '/kaggle/working/data/manifests'
]

manifest_dir = None
for dir_path in possible_manifest_dirs:
    # Check if directory exists and has manifest files
    if os.path.exists(dir_path):
        test_files = ['train_manifest.json', 'val_manifest.json', 'test_manifest.json']
        if any(os.path.exists(os.path.join(dir_path, f)) for f in test_files):
            manifest_dir = Path(dir_path)
            print(f"✓ Found manifests at: {manifest_dir}")
            break

if manifest_dir is None:
    print("✗ No manifest directory found!")
    print("\nSearching for manifest files...")
    !find /kaggle/working -name "*manifest*.json" 2>/dev/null | grep -v "\._" | head -10

In [ ]:
# If manifests not found, try to prepare them
if manifest_dir is None or not manifest_dir.exists():
    print("Attempting to prepare manifests...")
    
    # Check if preparation script exists
    prep_script = Path('/kaggle/working/scripts/prepare_raw_corpus_data.py')
    if prep_script.exists():
        print("Running data preparation script...")
        !python /kaggle/working/scripts/prepare_raw_corpus_data.py
        manifest_dir = Path('/kaggle/working/data/konkani-combined/manifests')
    else:
        print("⚠️  Preparation script not found.")
        print("Please ensure your dataset includes pre-prepared manifest files.")
        print("\nExpected structure:")
        print("  data/manifests/train.json")
        print("  data/manifests/val.json")
        print("  data/manifests/test.json")

In [ ]:
# Verify manifests and show dataset statistics
if manifest_dir and manifest_dir.exists():
    print("✓ Dataset manifests found:")
    print("=" * 60)
    
    total_duration = 0
    # Try both naming conventions
    manifest_files = [
        ('train_manifest.json', 'train.json'),
        ('val_manifest.json', 'val.json'),
        ('test_manifest.json', 'test.json')
    ]
    
    for primary_name, alt_name in manifest_files:
        manifest_path = manifest_dir / primary_name
        if not manifest_path.exists():
            manifest_path = manifest_dir / alt_name
        
        if manifest_path.exists():
            # Try loading as JSONL (one JSON per line) or JSON array
            data = []
            with open(manifest_path) as f:
                content = f.read().strip()
                try:
                    # Try as JSON array first
                    data = json.loads(content)
                except json.JSONDecodeError:
                    # Try as JSONL (one JSON object per line)
                    for line in content.split('\n'):
                        if line.strip():
                            data.append(json.loads(line))
            
            num_samples = len(data)
            
            # Calculate total duration if available
            duration = sum(item.get('duration', 0) for item in data)
            total_duration += duration
            
            display_name = manifest_path.name
            print(f"  {display_name:20s}: {num_samples:5,d} samples ({duration/3600:.1f}h)")
        else:
            print(f"  {primary_name:20s}: NOT FOUND")
    
    print("=" * 60)
    print(f"  Total Duration: {total_duration/3600:.1f} hours")
    print("✓ Ready to train!")
else:
    print("✗ ERROR: No manifest files found!")
    print("Cannot proceed with training.")

## Step 5: Configure Training (FIXED SETTINGS)

In [ ]:
# Training configuration with FIXES
import yaml

config = {
    'model': {
        'vocab_size': 200,
        'input_dim': 80,
        'd_model': 256,
        'encoder_layers': 12,
        'decoder_layers': 6,
        'num_heads': 4,
        'conv_kernel_size': 31,
        'dropout': 0.2
    },
    'training': {
        'learning_rate': 0.0003,      # 🔥 Increased from 0.0001
        'weight_decay': 0.0001,
        'grad_clip': 5.0,             # 🔥 Added gradient clipping
        'ctc_weight': 0.8,            # 🔥 CRITICAL FIX: was 0.3
        'batch_size': 4,              # 🔥 Increased for 2 GPUs (was 2)
        'gradient_accumulation_steps': 2,  # 🔥 Reduced for 2 GPUs (was 4)
        'mixed_precision': True,
        'num_epochs': 100,            # 🔥 More epochs
        'save_every': 5,
        'test_every': 5               # 🔥 Test every 5 epochs
    },
    'data': {
        'train_manifest': str(manifest_dir / 'train_manifest.json') if (manifest_dir / 'train_manifest.json').exists() else str(manifest_dir / 'train.json'),
        'val_manifest': str(manifest_dir / 'val_manifest.json') if (manifest_dir / 'val_manifest.json').exists() else str(manifest_dir / 'val.json'),
        'vocab_file': str(manifest_dir / 'vocab.json') if (manifest_dir / 'vocab.json').exists() else '/kaggle/working/konkani-10k/vocab.json',
        'num_workers': 2
    },
    'paths': {
        'checkpoint_dir': '/kaggle/working/checkpoints',
        'log_dir': '/kaggle/working/logs'
    },
    'device': 'cuda'
}

# Save config
os.makedirs('/kaggle/working/config', exist_ok=True)
with open('/kaggle/working/config/training_config_fixed.yaml', 'w') as f:
    yaml.dump(config, f)

print("✓ Training config saved with FIXES:")
print(f"  - CTC weight: {config['training']['ctc_weight']} (was 0.3)")
print(f"  - Learning rate: {config['training']['learning_rate']} (was 0.0001)")
print(f"  - Gradient clip: {config['training']['grad_clip']} (was None)")
print(f"  - Testing: Every {config['training']['test_every']} epochs")

## Step 6: Fix Manifest Audio Paths

In [ ]:
# Fix audio paths in manifests to point to Kaggle locations
print("Fixing audio paths in manifests...")

for manifest_name in ['train_manifest.json', 'val_manifest.json', 'test_manifest.json']:
    manifest_path = manifest_dir / manifest_name
    if not manifest_path.exists():
        manifest_path = manifest_dir / manifest_name.replace('_manifest', '')
    
    if manifest_path.exists():
        # Read manifest
        with open(manifest_path) as f:
            content = f.read().strip()
        
        # Parse as JSONL
        data = []
        for line in content.split('\n'):
            if line.strip():
                try:
                    data.append(json.loads(line))
                except:
                    pass
        
        # Fix paths
        fixed_count = 0
        for item in data:
            if 'audio_filepath' in item:
                old_path = item['audio_filepath']
                # Extract just the filename part after 'konkani-10k/audio/'
                if 'konkani-10k' in old_path or 'KonkaniRawSpeechCorpus' in old_path:
                    # Get the relative path from audio directory
                    if 'audio/' in old_path:
                        rel_path = old_path.split('audio/')[-1]
                    elif 'Data/' in old_path:
                        rel_path = old_path.split('Data/')[-1]
                    else:
                        rel_path = old_path.split('/')[-1]
                    
                    # Set new path
                    item['audio_filepath'] = f'/kaggle/working/konkani-10k/audio/Data/{rel_path}'
                    fixed_count += 1
        
        # Save fixed manifest
        with open(manifest_path, 'w') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        print(f"  ✓ {manifest_path.name}: Fixed {fixed_count}/{len(data)} paths")

print("\n✓ Manifest paths fixed!")

## Step 7: Fix Vocab File Format

In [ ]:
# Check and fix vocab file format
vocab_path = Path(config['data']['vocab_file'])

if vocab_path.exists():
    with open(vocab_path) as f:
        vocab_data = json.load(f)
    
    print(f"Current vocab keys: {list(vocab_data.keys())}")
    
    # Check if it needs conversion
    if 'idx2char' not in vocab_data:
        print("Converting vocab format...")
        
        # If it's a simple list/dict, convert it
        if isinstance(vocab_data, list):
            # List format: ['a', 'b', 'c', ...]
            char2idx = {char: idx for idx, char in enumerate(vocab_data)}
            idx2char = {idx: char for idx, char in enumerate(vocab_data)}
        elif 'char2idx' in vocab_data:
            # Has char2idx, create idx2char
            char2idx = vocab_data['char2idx']
            idx2char = {int(v): k for k, v in char2idx.items()}
        else:
            # Assume it's char2idx directly
            char2idx = vocab_data
            idx2char = {int(v): k for k, v in vocab_data.items()}
        
        # Save in correct format
        new_vocab = {
            'char2idx': char2idx,
            'idx2char': idx2char,
            'vocab_size': len(char2idx)
        }
        
        with open(vocab_path, 'w') as f:
            json.dump(new_vocab, f, ensure_ascii=False, indent=2)
        
        print(f"✓ Vocab converted! Size: {len(char2idx)} characters")
    else:
        print(f"✓ Vocab format is correct. Size: {vocab_data.get('vocab_size', len(vocab_data.get('char2idx', {})))}")
else:
    print(f"✗ Vocab file not found: {vocab_path}")

## Step 8: Check GPU Availability

In [ ]:
# Check available GPUs
import torch

num_gpus = torch.cuda.device_count()
print(f"Available GPUs: {num_gpus}")

for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

if num_gpus > 1:
    print(f"\n✓ Multi-GPU training available with {num_gpus} GPUs!")
    print("Note: The current training script uses single GPU.")
    print("For multi-GPU, the model would need DataParallel wrapping.")
else:
    print("\nSingle GPU training")

## Step 9: Setup Python Path and Enable Multi-GPU Training

In [ ]:
# Add working directory to Python path so imports work
import sys
sys.path.insert(0, '/kaggle/working')

print("Python path configured:")
print(f"  Working dir: /kaggle/working")
print(f"\nVerifying imports...")

try:
    from models.konkanivani_asr import create_konkanivani_model
    print("  ✓ models.konkanivani_asr")
except ImportError as e:
    print(f"  ✗ models.konkanivani_asr: {e}")

try:
    from data.audio_processing.audio_processor import AudioProcessor
    print("  ✓ data.audio_processing.audio_processor")
except ImportError as e:
    print(f"  ✗ data.audio_processing.audio_processor: {e}")

print("\n✓ Ready to train!")

In [ ]:
# Patch training script to enable multi-GPU DataParallel
import torch

print("Patching training script for multi-GPU support...")
print(f"Available GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

with open('/kaggle/working/training_scripts/train_konkanivani_asr.py', 'r') as f:
    script_content = f.read()

# Check if already patched
if 'torch.nn.DataParallel' not in script_content:
    print("\nApplying DataParallel patch...")
    
    # Patch 1: Enable DataParallel in __init__
    old_init = '        self.model = model.to(device)'
    new_init = '''        self.model = model.to(device)
        
        # Enable multi-GPU training with DataParallel
        self.is_multi_gpu = torch.cuda.device_count() > 1
        if self.is_multi_gpu:
            print(f"🚀 Using {torch.cuda.device_count()} GPUs with DataParallel")
            self.model = torch.nn.DataParallel(self.model)
        else:
            print("Using single GPU")'''
    
    script_content = script_content.replace(old_init, new_init)
    
    # Patch 2: Fix checkpoint saving to handle DataParallel
    old_save = "            'model_state_dict': self.model.state_dict(),"
    new_save = "            'model_state_dict': self.model.module.state_dict() if self.is_multi_gpu else self.model.state_dict(),"
    
    script_content = script_content.replace(old_save, new_save)
    
    # Write back
    with open('/kaggle/working/training_scripts/train_konkanivani_asr.py', 'w') as f:
        f.write(script_content)
    
    print("✓ Training script patched for multi-GPU support!")
    if torch.cuda.device_count() > 1:
        print(f"  Will use {torch.cuda.device_count()} GPUs with DataParallel")
        print(f"  Effective batch size: {config['training']['batch_size']} x {torch.cuda.device_count()} = {config['training']['batch_size'] * torch.cuda.device_count()}")
else:
    print("\n✓ Training script already has multi-GPU support")
    if torch.cuda.device_count() > 1:
        print(f"  Will use {torch.cuda.device_count()} GPUs")

In [ ]:
# Start training with individual arguments
train_manifest = config['data']['train_manifest']
val_manifest = config['data']['val_manifest']
vocab_file = config['data']['vocab_file']

!cd /kaggle/working && PYTHONPATH=/kaggle/working python training_scripts/train_konkanivani_asr.py \
    --train_manifest {train_manifest} \
    --val_manifest {val_manifest} \
    --vocab_file {vocab_file} \
    --batch_size {config['training']['batch_size']} \
    --num_epochs {config['training']['num_epochs']} \
    --learning_rate {config['training']['learning_rate']} \
    --weight_decay {config['training']['weight_decay']} \
    --dropout {config['model']['dropout']} \
    --ctc_weight {config['training']['ctc_weight']} \
    --save_every {config['training']['save_every']} \
    --checkpoint_dir {config['paths']['checkpoint_dir']} \
    --log_dir {config['paths']['log_dir']} \
    --d_model {config['model']['d_model']} \
    --encoder_layers {config['model']['encoder_layers']} \
    --decoder_layers {config['model']['decoder_layers']} \
    --gradient_accumulation_steps {config['training']['gradient_accumulation_steps']} \
    --mixed_precision \
    --device cuda

## Step 7: Monitor Progress

### Expected Timeline:
- **Epoch 1-10**: Blank prob 95-98% (learning basics)
- **Epoch 10-20**: Blank prob 80-90% (characters appearing)
- **Epoch 20-40**: Blank prob 50-80% ✅ **WORKING!**
- **Epoch 40-100**: Blank prob 30-50% (refinement)

In [ ]:
# Check test results
test_results_dir = Path('/kaggle/working/checkpoints')
test_files = sorted(test_results_dir.glob('test_results_epoch_*.json'))

if test_files:
    print("Test Results Summary:")
    print("=" * 80)
    for test_file in test_files:
        with open(test_file) as f:
            results = json.load(f)
            epoch = results.get('epoch', '?')
            blank_prob = results.get('avg_blank_prob', 0)
            status = '✅ WORKING!' if blank_prob < 80 else '❌ Not yet'
            print(f"Epoch {epoch:3d}: Blank prob {blank_prob:5.1f}% - {status}")
else:
    print("No test results yet. Check back after epoch 5.")

## Step 8: Download Best Checkpoint

In [ ]:
# Find best checkpoint (lowest validation loss)
checkpoint_dir = Path('/kaggle/working/checkpoints')
checkpoints = sorted(checkpoint_dir.glob('checkpoint_epoch_*.pt'))

if checkpoints:
    best_ckpt = None
    best_val_loss = float('inf')
    
    for ckpt_path in checkpoints:
        ckpt = torch.load(ckpt_path, map_location='cpu')
        val_loss = ckpt.get('val_loss', float('inf'))
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_ckpt = ckpt_path
    
    print(f"Best checkpoint: {best_ckpt.name}")
    print(f"Validation loss: {best_val_loss:.4f}")
    
    # Copy to best_model.pt
    import shutil
    shutil.copy(best_ckpt, checkpoint_dir / 'best_model.pt')
    print("✓ Saved as best_model.pt")
else:
    print("No checkpoints found yet")

In [ ]:
# Create download link
from IPython.display import FileLink

print("Download your trained model:")
FileLink('/kaggle/working/checkpoints/best_model.pt')

## Step 9: Quick Test

In [ ]:
# Test the best model on a few samples
!python /kaggle/working/scripts/test_best_model.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pt \
    --max_files 10

## Step 10: Generate Training Visualization

In [ ]:
# Generate comprehensive training metrics visualization
!python /kaggle/working/scripts/generate_training_visualization.py \
    --log /kaggle/working/logs/training.log \
    --output /kaggle/working/training_metrics.png \
    --title "Konkani ASR"

In [ ]:
# Display the visualization
from IPython.display import Image, display
import os

if os.path.exists('/kaggle/working/training_metrics.png'):
    print("✓ Training Visualization:")
    display(Image('/kaggle/working/training_metrics.png'))
else:
    print("✗ Visualization not generated yet. Run after training completes.")

In [ ]:
# Download link for the visualization
from IPython.display import FileLink

print("Download training visualization:")
FileLink('/kaggle/working/training_metrics.png')

## Summary

### Key Fixes Applied:
1. ✅ CTC weight: 0.3 → 0.8 (critical for transcription)
2. ✅ Learning rate: 0.0001 → 0.0003 (faster learning)
3. ✅ Added gradient clipping: 5.0 (stability)
4. ✅ Full dataset: 21h → 88h (4x more data)
5. ✅ Periodic testing: Monitor every 5 epochs

### Expected Results:
- Model should start working by epoch 20-30
- Blank probability should drop below 80%
- Transcriptions should be recognizable
- Final CER should be 20-40%

### Next Steps:
1. Download best_model.pt
2. Download training_metrics.png
3. Test locally on your audio files
4. Deploy for production use